# Imputación usando MissForest

In [ ]:
# ################################################################################### #
# Script used for the imputation of the missing values of the dataset by means of the #
# technique MissForest. Reference:                                                    #
#                                                                                     #
#           Daniel J. Stekhoven and Peter Bühlmann. MissForest - nonparametric        #
#           missing value imputation for mixed-type data. 2011, 28, 1, 112-118.       #
#           DOI: 10.1093/bioinformatics/btr597.                                       #
#                                                                                     #
# ################################################################################### #    

## Idea general


1. Imputar numéricas + categóricas ordinales con “MissForest” (IterativeImputer + árboles)

2. Para categóricas nominales, NO imputarlas como números ni como one-hot continuo; imputarlas como categoría (más simple y estable).

Es necesario hacer esta separación porque:

- **Las ordinales sí tienen orden** → tiene sentido imputarlas como números y luego redondear

- **Las nominales no tienen orden** → imputarlas como números mete orden falso; imputarlas como one-hot con modelos de regresión deja dummies inconsistentes.

In [2]:
# Import basics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [3]:
df = pd.read_csv('data/full_dataset.csv')
df

,genero,edad,etnia_1,etnia_3,mes_examen,pais_nacimiento,tamano_hogar,peso_entrevista,peso_examen,estrato,psu,ratio_pobreza,peso_dieta,total_suplementos,total_antiacidos,uso_suplementos,uso_antiacidos,estado_equilibrio,romberg_1_seg,romberg_2_seg,romberg_3_seg,romberg_4_seg,brazo,manguito,sistolica_1,diastolica_1,sistolica_2,diastolica_2,sistolica_3,diastolica_3,pulso_1,pulso_2,pulso_3,estado_antropo,peso_kg,talla_cm,imc,pierna_cm,brazo_cm,perim_brazo,cintura_cm,cadera_cm,estado_higado,sonda,med_validas,intentos,rigidez_kpa,iqr_rigidez,ratio_iqrm,cap_dbm,iqr_cap,albumina_orina,albmina_si,com_albumina,creatinina_orina,creatinina_si,ratio_albcre,peso_sangre,blancos,_linfocitos,_monocitos,_neutrofilos,_eosinofilos,_basofilos,linfocitos_abs,monocitos_abs,neutrofilos_abs,eosinofilos_abs,basofilos_abs,rojos,hemoglobina,hematocrito,vcm,chcm,hcm,rdw,plaquetas,vpm,nrbc,peso_ayuno,trigliceridos,trigli_si,ldl,ldl_si,ldl_martin,ldl_martin_si,ldl_nih,ldl_nih_si,hdl,hdl_si,proteina_c,com_pcr,plomo,plomo_si,cadmio,cadmio_si,com_cadmio,mercurio,mercurio_si,com_mercurio,selenio,selenio_si,manganeso,manganeso_si,movilidad,neurologico,osteoporosis,equilibrio
0,1.0,43.0,5.0,6.0,2.0,2.0,4.0,50055.450807,54374.463898,173.0,2.0,5.00,61366.555827,5.397605e-79,5.397605e-79,0.0,0.0,1.0,15.0,15.0,30.0,13.0,b'R',4.0,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0,1.0,86.9,179.5,27.0,42.8,42.0,35.7,98.3,102.9,1.0,b'M',10.0,11.0,9.5,2.3,24.2,380.0,19.0,23.12,23.12,5.397605e-79,136.0,12022.4,17.00,56042.129410,4.7,40.8,6.8,46.1,5.2,1.2,1.9,0.3,2.2,2.000000e-01,1.000000e-01,5.19,15.7,45.3,87.4,34.6,30.2,12.9,259.0,7.8,3.000000e-01,1.200253e+05,153.0,1.727,188.0,4.862,190.0,4.913,191.0,4.939,45.0,1.16,1.78,5.397605e-79,2.810,0.136,0.117,1.041,5.397605e-79,1.01,5.04,5.397605e-79,189.2,2.40,10.94,199.13,0.0,1.0,1.0,1.0
1,1.0,66.0,3.0,3.0,2.0,1.0,2.0,29087.450605,34084.721548,173.0,2.0,5.00,34638.056480,5.397605e-79,5.397605e-79,0.0,0.0,1.0,15.0,15.0,30.0,25.0,b'R',4.0,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0,1.0,101.8,174.2,33.5,38.5,38.7,33.7,114.7,112.4,1.0,b'XL',12.0,17.0,3.9,0.8,20.5,345.0,29.0,4.25,4.25,5.397605e-79,64.0,5657.6,6.64,37435.705647,6.3,25.4,14.2,57.3,2.5,0.7,1.6,0.9,3.6,2.000000e-01,5.397605e-79,4.59,15.2,43.9,95.8,34.5,33.0,12.8,221.0,8.5,1.000000e-01,5.397605e-79,86.0,0.971,137.0,3.543,135.0,3.491,139.0,3.595,60.0,1.55,2.03,5.397605e-79,2.040,0.099,0.313,2.785,5.397605e-79,9.64,48.10,5.397605e-79,192.3,2.44,7.74,140.88,0.0,1.0,1.0,1.0
2,2.0,44.0,2.0,2.0,1.0,2.0,7.0,80062.674301,81196.277992,174.0,1.0,1.41,84728.261560,1.000000e+00,5.397605e-79,1.0,0.0,1.0,15.0,15.0,30.0,2.0,b'R',4.0,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0,1.0,69.4,152.9,29.7,38.5,35.5,36.3,93.5,98.0,1.0,b'M',10.0,10.0,7.6,1.4,18.4,255.0,90.0,12.43,12.43,5.397605e-79,157.0,13878.8,7.92,85328.844519,5.7,29.9,5.5,58.5,5.3,0.9,1.7,0.3,3.3,3.000000e-01,1.000000e-01,4.86,13.8,40.1,82.5,34.4,28.3,13.3,235.0,9.1,1.000000e-01,1.450908e+05,375.0,4.234,63.0,1.629,90.0,2.327,78.0,2.017,49.0,1.27,5.62,5.397605e-79,0.399,0.019,0.270,2.402,5.397605e-79,0.55,2.74,5.397605e-79,160.5,2.04,11.93,217.15,1.0,1.0,1.0,1.0
3,1.0,34.0,1.0,1.0,1.0,1.0,3.0,30995.282610,39988.452940,179.0,1.0,1.33,82013.365563,2.000000e+00,5.397605e-79,1.0,0.0,1.0,15.0,15.0,30.0,30.0,b'R',4.0,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0,1.0,90.6,173.3,30.2,42.8,36.2,35.7,106.1,110.6,1.0,b'M',10.0,11.0,3.1,0.3,9.7,163.0,56.0,4.60,4.60,5.397605e-79,113.0,9989.2,4.07,44526.214135,5.5,54.4,7.5,35.6,2.1,0.5,3.0,0.4,2.0,1.000000e-01,5.397605e-79,5.06,15.4,43.5,86.1,35.3,30.4,12.8,250.0,7.4,5.397605e-79,8.259962e+04,142.0,1.603,109.0,2.819,111.0,2.870,112.0,2.896,46.0,1.19,1.05,5.397605e-79,1.810,0.087,0.190,1.690,5.397605e-79,0.34,1.70,5.397605e-79,183.3,2.33,10.94,199.13,0.0,1.0,1.0,1.0
4,1.0,51.0,3.0,3.0,1.0,1.0,4.0,41925.463225,51305.024430,174.0,2.0,5.00,57902.412643,1.000000e+00,5.397605e-79,1.0,0.0,1.0,15.0,15.0,30.0,5.0,b'R',3.0,99.0,69.0,110.0,68.0,123.0,67.0,78.0,82.0,79.0,1.0,76.7,177.3,24.4,41.0,

In [4]:
# Para asegurar que el imputador funcione bien convertimos los valores faltantes, representados en blanco, a NaN
df.replace("", np.nan, inplace=True)
df

,genero,edad,etnia_1,etnia_3,mes_examen,pais_nacimiento,tamano_hogar,peso_entrevista,peso_examen,estrato,psu,ratio_pobreza,peso_dieta,total_suplementos,total_antiacidos,uso_suplementos,uso_antiacidos,estado_equilibrio,romberg_1_seg,romberg_2_seg,romberg_3_seg,romberg_4_seg,brazo,manguito,sistolica_1,diastolica_1,sistolica_2,diastolica_2,sistolica_3,diastolica_3,pulso_1,pulso_2,pulso_3,estado_antropo,peso_kg,talla_cm,imc,pierna_cm,brazo_cm,perim_brazo,cintura_cm,cadera_cm,estado_higado,sonda,med_validas,intentos,rigidez_kpa,iqr_rigidez,ratio_iqrm,cap_dbm,iqr_cap,albumina_orina,albmina_si,com_albumina,creatinina_orina,creatinina_si,ratio_albcre,peso_sangre,blancos,_linfocitos,_monocitos,_neutrofilos,_eosinofilos,_basofilos,linfocitos_abs,monocitos_abs,neutrofilos_abs,eosinofilos_abs,basofilos_abs,rojos,hemoglobina,hematocrito,vcm,chcm,hcm,rdw,plaquetas,vpm,nrbc,peso_ayuno,trigliceridos,trigli_si,ldl,ldl_si,ldl_martin,ldl_martin_si,ldl_nih,ldl_nih_si,hdl,hdl_si,proteina_c,com_pcr,plomo,plomo_si,cadmio,cadmio_si,com_cadmio,mercurio,mercurio_si,com_mercurio,selenio,selenio_si,manganeso,manganeso_si,movilidad,neurologico,osteoporosis,equilibrio
0,1.0,43.0,5.0,6.0,2.0,2.0,4.0,50055.450807,54374.463898,173.0,2.0,5.00,61366.555827,5.397605e-79,5.397605e-79,0.0,0.0,1.0,15.0,15.0,30.0,13.0,b'R',4.0,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0,1.0,86.9,179.5,27.0,42.8,42.0,35.7,98.3,102.9,1.0,b'M',10.0,11.0,9.5,2.3,24.2,380.0,19.0,23.12,23.12,5.397605e-79,136.0,12022.4,17.00,56042.129410,4.7,40.8,6.8,46.1,5.2,1.2,1.9,0.3,2.2,2.000000e-01,1.000000e-01,5.19,15.7,45.3,87.4,34.6,30.2,12.9,259.0,7.8,3.000000e-01,1.200253e+05,153.0,1.727,188.0,4.862,190.0,4.913,191.0,4.939,45.0,1.16,1.78,5.397605e-79,2.810,0.136,0.117,1.041,5.397605e-79,1.01,5.04,5.397605e-79,189.2,2.40,10.94,199.13,0.0,1.0,1.0,1.0
1,1.0,66.0,3.0,3.0,2.0,1.0,2.0,29087.450605,34084.721548,173.0,2.0,5.00,34638.056480,5.397605e-79,5.397605e-79,0.0,0.0,1.0,15.0,15.0,30.0,25.0,b'R',4.0,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0,1.0,101.8,174.2,33.5,38.5,38.7,33.7,114.7,112.4,1.0,b'XL',12.0,17.0,3.9,0.8,20.5,345.0,29.0,4.25,4.25,5.397605e-79,64.0,5657.6,6.64,37435.705647,6.3,25.4,14.2,57.3,2.5,0.7,1.6,0.9,3.6,2.000000e-01,5.397605e-79,4.59,15.2,43.9,95.8,34.5,33.0,12.8,221.0,8.5,1.000000e-01,5.397605e-79,86.0,0.971,137.0,3.543,135.0,3.491,139.0,3.595,60.0,1.55,2.03,5.397605e-79,2.040,0.099,0.313,2.785,5.397605e-79,9.64,48.10,5.397605e-79,192.3,2.44,7.74,140.88,0.0,1.0,1.0,1.0
2,2.0,44.0,2.0,2.0,1.0,2.0,7.0,80062.674301,81196.277992,174.0,1.0,1.41,84728.261560,1.000000e+00,5.397605e-79,1.0,0.0,1.0,15.0,15.0,30.0,2.0,b'R',4.0,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0,1.0,69.4,152.9,29.7,38.5,35.5,36.3,93.5,98.0,1.0,b'M',10.0,10.0,7.6,1.4,18.4,255.0,90.0,12.43,12.43,5.397605e-79,157.0,13878.8,7.92,85328.844519,5.7,29.9,5.5,58.5,5.3,0.9,1.7,0.3,3.3,3.000000e-01,1.000000e-01,4.86,13.8,40.1,82.5,34.4,28.3,13.3,235.0,9.1,1.000000e-01,1.450908e+05,375.0,4.234,63.0,1.629,90.0,2.327,78.0,2.017,49.0,1.27,5.62,5.397605e-79,0.399,0.019,0.270,2.402,5.397605e-79,0.55,2.74,5.397605e-79,160.5,2.04,11.93,217.15,1.0,1.0,1.0,1.0
3,1.0,34.0,1.0,1.0,1.0,1.0,3.0,30995.282610,39988.452940,179.0,1.0,1.33,82013.365563,2.000000e+00,5.397605e-79,1.0,0.0,1.0,15.0,15.0,30.0,30.0,b'R',4.0,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0,1.0,90.6,173.3,30.2,42.8,36.2,35.7,106.1,110.6,1.0,b'M',10.0,11.0,3.1,0.3,9.7,163.0,56.0,4.60,4.60,5.397605e-79,113.0,9989.2,4.07,44526.214135,5.5,54.4,7.5,35.6,2.1,0.5,3.0,0.4,2.0,1.000000e-01,5.397605e-79,5.06,15.4,43.5,86.1,35.3,30.4,12.8,250.0,7.4,5.397605e-79,8.259962e+04,142.0,1.603,109.0,2.819,111.0,2.870,112.0,2.896,46.0,1.19,1.05,5.397605e-79,1.810,0.087,0.190,1.690,5.397605e-79,0.34,1.70,5.397605e-79,183.3,2.33,10.94,199.13,0.0,1.0,1.0,1.0
4,1.0,51.0,3.0,3.0,1.0,1.0,4.0,41925.463225,51305.024430,174.0,2.0,5.00,57902.412643,1.000000e+00,5.397605e-79,1.0,0.0,1.0,15.0,15.0,30.0,5.0,b'R',3.0,99.0,69.0,110.0,68.0,123.0,67.0,78.0,82.0,79.0,1.0,76.7,177.3,24.4,41.0,

In [5]:
df.shape

(2527, 108)

In [6]:
df.columns

Index(['genero', 'edad', 'etnia_1', 'etnia_3', 'mes_examen', 'pais_nacimiento',
       'tamano_hogar', 'peso_entrevista', 'peso_examen', 'estrato',
       ...
       'mercurio_si', 'com_mercurio', 'selenio', 'selenio_si', 'manganeso',
       'manganeso_si', 'movilidad', 'neurologico', 'osteoporosis',
       'equilibrio'],
      dtype='object', length=108)

### Esta es la celda que usaríamos si antes hiciesemos split en train/test

In [26]:
# Funciones para la imputación
def _missForest_fit(data, num_estimators, m_depth, m_samples_leaf, max_iter,
                    min_value=None, max_value=None, random_state=1234):
    tree = ExtraTreesRegressor(n_estimators=num_estimators,
                               random_state=random_state,
                               max_depth=m_depth,
                               min_samples_leaf=m_samples_leaf)
    imputer = IterativeImputer(estimator=tree,
                               random_state=random_state,
                               max_iter=max_iter,
                               min_value=min_value,
                               max_value=max_value,
                               imputation_order='ascending',
                               initial_strategy='mean',
                               verbose=2)
    imputer.fit(data)
    return imputer

def _missForest_transform(imputer, data):
    return imputer.transform(data)


def imputacion_pipeline(train_num, test_num,num_estimators, 
                        m_depth, m_samples_leaf,
                        max_iter, random_state=1234,
                        guardar_imputador=True):

    # Obtener min/max automáticamente por nombre de variable
    min_values_num = [min_values_dict[col] for col in train_num.columns]
    max_values_num = [max_values_dict[col] for col in train_num.columns]

    min_values_all = [min_values_dict[col] for col in list(train_num.columns)]
    max_values_all = [max_values_dict[col] for col in list(train_num.columns)]

    
    # Imputación de numéricas
    imputer_num = _missForest_fit(train_num, num_estimators, m_depth, m_samples_leaf,
                                  max_iter, min_value=min_values_num,
                                  max_value=max_values_num,
                                  random_state=random_state)
    train_num_imp = pd.DataFrame(_missForest_transform(imputer_num, train_num),
                                 columns=train_num.columns,
                                 index=train_num.index)
    test_num_imp = pd.DataFrame(_missForest_transform(imputer_num, test_num),
                                columns=test_num.columns,
                                index=test_num.index)

    # Unir con categóricas
    train_all = pd.concat([train_num_imp], axis=1)
    test_all = pd.concat([test_num_imp], axis=1)

    # Imputación conjunta
    imputer_all = _missForest_fit(train_all, num_estimators, m_depth, m_samples_leaf,
                                  max_iter, min_value=min_values_all,
                                  max_value=max_values_all,
                                  random_state=random_state)
    train_all_imp = _missForest_transform(imputer_all, train_all)
    test_all_imp = _missForest_transform(imputer_all, test_all)

    # Control de variables categóricas
    for i in range(train_num.shape[1], train_all.shape[1]):
        train_all_imp[:, i] = np.clip(train_all_imp[:, i], min_values_all[i], max_values_all[i])
        test_all_imp[:, i] = np.clip(test_all_imp[:, i], min_values_all[i], max_values_all[i])

    train_final = pd.DataFrame(train_all_imp, columns=train_all.columns, index=train_all.index)
    test_final = pd.DataFrame(test_all_imp, columns=test_all.columns, index=test_all.index)

    if guardar_imputador:
        joblib.dump(imputer_all, 'imputador_missforest.pkl')
        print("Imputador guardado como 'imputador_missforest.pkl'")

    return train_final, test_final

## Flujo general del código

**1. Definición de rangos permitidos (min_value y max_value)**

El imputador MissForest, a través de IterativeImputer, puede imputar valores fuera de los rangos clínicos reales si no se le ponen restricciones.  Por ello, se definen dos diccionarios: min_values_dict (mapeo de cada variable a su valor mínimo aceptado) y max_values_dict (mapeo de cada variable, su valor máximo aceptado). Esto se hace por nombre de variable para evitar errores si se cambia el orden de las columnas.

**2. Entrenamiento del imputador MissForest solo en datos de entrenamiento**

Este paso es necesario para evitar data leakage, el imputador se entrena solo con datos de entrenamiento. Luego se aplica al conjunto de test sin volver a ajustar.

**3. Aplicación del imputador a test**

El conjunto de test debe representar datos nunca vistos. Solo podemos usar el imputador previamente entrenado en train, ya que en producción no conoceremos la distribución real del test.

**4. Unión de variables categóricas (no imputadas todavía)**

Las categóricas también deben imputarse si tienen valores faltantes. Por eficiencia y consistencia, se hace una segunda imputación sobre todo el conjunto combinado (números ya imputados + categóricas originales).

**5. Imputación conjunta de numéricas + categóricas**

Esta segunda imputación considera interacciones entre variables numéricas y categóricas. Se usan min_values_all y max_values_all que incluyen las restricciones de todas las variables.

**6. Clipping y redondeo de variables categóricas**

Aunque se hayan definido rangos, MissForest puede producir pequeños decimales. Las variables categóricas deben ser redondeadas al entero más cercano y convertidas a int.

**7. Guardado del imputador para producción**

Guardar el imputador permite reutilizarlo en producción (por ejemplo, en una API Flask), garantizando que los datos futuros se traten exactamente igual que los del entrenamiento.

### Esta es la celda que utilizamos sin dividir en train/test 

In [31]:
# Funciones para la imputación
def _missForest_fit(data, num_estimators, m_depth, m_samples_leaf, max_iter,
                    min_value=None, max_value=None, random_state=1234):
    tree = ExtraTreesRegressor(
        n_estimators=num_estimators,
        random_state=random_state,
        max_depth=m_depth,
        min_samples_leaf=m_samples_leaf
    )
    imputer = IterativeImputer(
        estimator=tree,
        random_state=random_state,
        max_iter=max_iter,
        min_value=min_value,
        max_value=max_value,
        imputation_order='ascending',
        initial_strategy='mean',
        verbose=2
    )
    imputer.fit(data)
    return imputer

def _missForest_transform(imputer, data):
    return imputer.transform(data)


def imputacion_pipeline_all(data_num, num_estimators, 
                            m_depth, m_samples_leaf,
                            max_iter, random_state=1234,
                            guardar_imputador=True):

    # Obtener min/max automáticamente por nombre de variable
    min_values_num = [min_values_dict[col] for col in data_num.columns]
    max_values_num = [max_values_dict[col] for col in data_num.columns]

    min_values_all = [min_values_dict[col] for col in list(data_num.columns)]
    max_values_all = [max_values_dict[col] for col in list(data_num.columns)]

    # Imputación de numéricas
    imputer_num = _missForest_fit(
        data_num, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_num,
        max_value=max_values_num,
        random_state=random_state
    )

    data_num_imp = pd.DataFrame(
        _missForest_transform(imputer_num, data_num),
        columns=data_num.columns,
        index=data_num.index
    )

    # Unir con categóricas (ordinales)
    data_all = pd.concat([data_num_imp], axis=1)

    # Imputación conjunta
    imputer_all = _missForest_fit(
        data_all, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_all,
        max_value=max_values_all,
        random_state=random_state
    )

    data_all_imp = _missForest_transform(imputer_all, data_all)

    # Control de variables categóricas (ordinales)
    for i in range(data_num.shape[1], data_all.shape[1]):
        data_all_imp[:, i] = np.clip(data_all_imp[:, i], min_values_all[i], max_values_all[i])

    data_final = pd.DataFrame(data_all_imp, columns=data_all.columns, index=data_all.index)

    if guardar_imputador:
        joblib.dump(imputer_all, 'imputador_missforest.pkl')
        print("Imputador guardado como 'imputador_missforest.pkl'")

    return data_final

## Los rangos min_value y max_value deben calcularse exclusivamente sobre el conjunto de train

Así evitamos **data leakage**. Si extraemos los valores mínimos y máximos de todo el dataset, estarías indirectamente "mirando" los datos de test durante el entrenamiento del imputador → esto contamina el modelo.

**Coherencia con la imputación**: MissForest usa los rangos para limitar las predicciones durante el proceso iterativo. Si esos rangos provienen de test, estás condicionando la forma en la que el modelo "aprende" los valores del train.

**Producción realista**: En un entorno real, nunca dispones del test en el momento de entrenar el modelo. Así que los rangos utilizados deben derivarse únicamente de los datos conocidos (train).

**¿Qué pasa si en test hay valores fuera de los rangos de train?**

Esto no debería ocurrir si: hemos realizado una buena división estratificada y la muestra de train es representativa.

Pero si ocurre, no debemos ajustar los rangos. En su lugar, podemos:

- Aplicar un clip() posterior para limitar valores extremos en test.

- Analizar si el test contiene poblaciones diferentes y considerar un rediseño del muestreo.

In [9]:
min_values_dict = df.min().to_dict()
max_values_dict = df.max().to_dict()

In [10]:
min_values_dict

{'genero': 1.0,
 'edad': 20.0,
 'etnia_1': 1.0,
 'etnia_3': 1.0,
 'mes_examen': 1.0,
 'pais_nacimiento': 1.0,
 'tamano_hogar': 1.0,
 'peso_entrevista': 4584.463196,
 'peso_examen': 5424.587296,
 'estrato': 173.0,
 'psu': 1.0,
 'ratio_pobreza': 5.397605346934028e-79,
 'peso_dieta': 5.397605346934028e-79,
 'total_suplementos': 5.397605346934028e-79,
 'total_antiacidos': 5.397605346934028e-79,
 'uso_suplementos': 0.0,
 'uso_antiacidos': 0.0,
 'estado_equilibrio': 1.0,
 'romberg_1_seg': 1.0,
 'romberg_2_seg': 5.397605346934028e-79,
 'romberg_3_seg': 1.0,
 'romberg_4_seg': 5.397605346934028e-79,
 'brazo': "b''",
 'manguito': 2.0,
 'sistolica_1': 75.0,
 'diastolica_1': 44.0,
 'sistolica_2': 65.0,
 'diastolica_2': 39.0,
 'sistolica_3': 62.0,
 'diastolica_3': 26.0,
 'pulso_1': 35.0,
 'pulso_2': 35.0,
 'pulso_3': 36.0,
 'estado_antropo': 1.0,
 'peso_kg': 27.9,
 'talla_cm': 133.0,
 'imc': 11.1,
 'pierna_cm': 24.9,
 'brazo_cm': 29.1,
 'perim_brazo': 16.0,
 'cintura_cm': 62.4,
 'cadera_cm': 74.5,


In [11]:
max_values_dict

{'genero': 2.0,
 'edad': 69.0,
 'etnia_1': 5.0,
 'etnia_3': 7.0,
 'mes_examen': 2.0,
 'pais_nacimiento': 2.0,
 'tamano_hogar': 7.0,
 'peso_entrevista': 170968.343177,
 'peso_examen': 227108.296958,
 'estrato': 187.0,
 'psu': 2.0,
 'ratio_pobreza': 5.0,
 'peso_dieta': 408505.813642,
 'total_suplementos': 99.0,
 'total_antiacidos': 77.0,
 'uso_suplementos': 1.0,
 'uso_antiacidos': 1.0,
 'estado_equilibrio': 4.0,
 'romberg_1_seg': 15.0,
 'romberg_2_seg': 15.0,
 'romberg_3_seg': 30.0,
 'romberg_4_seg': 30.0,
 'brazo': "b'R'",
 'manguito': 5.0,
 'sistolica_1': 211.0,
 'diastolica_1': 131.0,
 'sistolica_2': 208.0,
 'diastolica_2': 129.0,
 'sistolica_3': 204.0,
 'diastolica_3': 134.0,
 'pulso_1': 129.0,
 'pulso_2': 129.0,
 'pulso_3': 128.0,
 'estado_antropo': 4.0,
 'peso_kg': 224.1,
 'talla_cm': 198.1,
 'imc': 74.8,
 'pierna_cm': 49.8,
 'brazo_cm': 49.2,
 'perim_brazo': 61.0,
 'cintura_cm': 177.2,
 'cadera_cm': 182.5,
 'estado_higado': 1.0,
 'sonda': "b'XL'",
 'med_validas': 30.0,
 'intentos'

## ¿Por qué se realiza la imputación en 2 pasos: primero variables numéricas y luego numéricas+categóricas?

Imputamos primero solo las variables numéricas por:

1. Mejor estabilidad del imputador numérico

Los algoritmos como MissForest (basado aquí en IterativeImputer + ExtraTreesRegressor) pueden beneficiarse de entrenarse primero solo con variables numéricas. Esto:

- Reduce el ruido introducido por las categóricas codificadas como números

- Mejora la estimación de patrones numéricos y relaciones entre variables continuas.

- Permite detectar y excluir variables problemáticas antes de mezclar tipos de datos (por ejemplo, variables constantes o con distribución extraña).

2. Control independiente de los rangos

A nivel práctico, se puede definir una lista más sencilla de min_values y max_values solo para variables numéricas y verificar su comportamiento antes de introducir restricciones adicionales para categóricas.

**¿Por qué luego se imputan conjuntamente las numéricas (ya imputadas) y las categóricas?**

Una vez imputadas las numéricas, se hace una segunda imputación conjunta de: numéricas_imputadas + categóricas_con_missing


1. Mejora de la imputación de categóricas

Las variables categóricas se imputan mejor si se aprovecha la información de las variables numéricas completas. Por ejemplo:

La paridad puede estar relacionada con la edad materna o con el IMC.

La codificación de Doppler puede depender del flujo sanguíneo cuantificado en variables continuas.

2. Coherencia entre variables

Permite capturar dependencias cruzadas entre tipos de variables (ej. una categoría que suele coincidir con valores altos en una variable continua).

3. No se reimputan las numéricas

En esta segunda etapa, las numéricas ya imputadas no se modifican, pero participan como predictores para imputar las categóricas. (El IterativeImputer respeta valores ya imputados si se le pasan como entrada sin NaNs).

In [27]:
# Diferenciar variables numéricas y categóricas nominales y ordinales
cat_cols_all = ['ciclo', 'estado', 'genero', 'etnia_1', 'etnia_3', 'mes_examen', 'pais_nacimiento', 'uso_suplementos', 'uso_antiacidos', 'estado_equilibrio', 'estado_romberg', 'mareos', 'desmayos', 'convulsiones', 'dificultad_caminar', 'caidas', 'fracturas', 'test_densidad', 'med_osteoporosis', 'diag_osteoporosis', 'romberg_1_res', 'romberg_2_res', 'romberg_3_res', 'romberg_4_res', 'brazo', 'manguito', 'estado_antropo', 'estado_higado', 'sonda', 'com_albumina', 'com_creatinina', 'com_pcr', 'com_plomo', 'com_cadmio', 'com_mercurio', 'com_selenio', 'com_manganeso', 'movilidad', 'neurologico', 'osteoporosis', 'equilibrio']
cat_cols_nom = [col for col in cat_cols_all if col in df.columns]
cat_cols_ord = []
num_cols = [col for col in df.columns if col not in cat_cols_all]

# División
df_num = df[num_cols]
# df_cat_ord = df[cat_cols_ord]
df_cat_nom = df[cat_cols_nom]

Los rangos tienen que estar ordenados en función de lo que "entre" a la imputación.

1. Imputación de las variables numéricas (train_num): rangos min/max en este orden

2. Imputación de las variables numéricas + categóricas: aquí lo que "entra" a la imputación es train_all = pd.concat([train_num, train_cat], axis=1). Por lo tanto, el orden correcto de los rangos min/max será primero todas las numéricas y luego todas las categóricas. 

In [15]:
df_num

,edad,tamano_hogar,peso_entrevista,peso_examen,estrato,psu,ratio_pobreza,peso_dieta,total_suplementos,total_antiacidos,romberg_1_seg,romberg_2_seg,romberg_3_seg,romberg_4_seg,sistolica_1,diastolica_1,sistolica_2,diastolica_2,sistolica_3,diastolica_3,pulso_1,pulso_2,pulso_3,peso_kg,talla_cm,imc,pierna_cm,brazo_cm,perim_brazo,cintura_cm,cadera_cm,med_validas,intentos,rigidez_kpa,iqr_rigidez,ratio_iqrm,cap_dbm,iqr_cap,albumina_orina,albmina_si,creatinina_orina,creatinina_si,ratio_albcre,peso_sangre,blancos,_linfocitos,_monocitos,_neutrofilos,_eosinofilos,_basofilos,linfocitos_abs,monocitos_abs,neutrofilos_abs,eosinofilos_abs,basofilos_abs,rojos,hemoglobina,hematocrito,vcm,chcm,hcm,rdw,plaquetas,vpm,nrbc,peso_ayuno,trigliceridos,trigli_si,ldl,ldl_si,ldl_martin,ldl_martin_si,ldl_nih,ldl_nih_si,hdl,hdl_si,proteina_c,plomo,plomo_si,cadmio,cadmio_si,mercurio,mercurio_si,selenio,selenio_si,manganeso,manganeso_si
0,43.0,4.0,50055.450807,54374.463898,173.0,2.0,5.00,61366.555827,5.397605e-79,5.397605e-79,15.0,15.0,30.0,13.0,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0,86.9,179.5,27.0,42.8,42.0,35.7,98.3,102.9,10.0,11.0,9.5,2.3,24.2,380.0,19.0,23.12,23.12,136.0,12022.4,17.00,56042.129410,4.7,40.8,6.8,46.1,5.2,1.2,1.9,0.3,2.2,2.000000e-01,1.000000e-01,5.19,15.7,45.3,87.4,34.6,30.2,12.9,259.0,7.8,3.000000e-01,1.200253e+05,153.0,1.727,188.0,4.862,190.0,4.913,191.0,4.939,45.0,1.16,1.78,2.810,0.136,0.117,1.041,1.01,5.04,189.2,2.40,10.94,199.13
1,66.0,2.0,29087.450605,34084.721548,173.0,2.0,5.00,34638.056480,5.397605e-79,5.397605e-79,15.0,15.0,30.0,25.0,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0,101.8,174.2,33.5,38.5,38.7,33.7,114.7,112.4,12.0,17.0,3.9,0.8,20.5,345.0,29.0,4.25,4.25,64.0,5657.6,6.64,37435.705647,6.3,25.4,14.2,57.3,2.5,0.7,1.6,0.9,3.6,2.000000e-01,5.397605e-79,4.59,15.2,43.9,95.8,34.5,33.0,12.8,221.0,8.5,1.000000e-01,5.397605e-79,86.0,0.971,137.0,3.543,135.0,3.491,139.0,3.595,60.0,1.55,2.03,2.040,0.099,0.313,2.785,9.64,48.10,192.3,2.44,7.74,140.88
2,44.0,7.0,80062.674301,81196.277992,174.0,1.0,1.41,84728.261560,1.000000e+00,5.397605e-79,15.0,15.0,30.0,2.0,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0,69.4,152.9,29.7,38.5,35.5,36.3,93.5,98.0,10.0,10.0,7.6,1.4,18.4,255.0,90.0,12.43,12.43,157.0,13878.8,7.92,85328.844519,5.7,29.9,5.5,58.5,5.3,0.9,1.7,0.3,3.3,3.000000e-01,1.000000e-01,4.86,13.8,40.1,82.5,34.4,28.3,13.3,235.0,9.1,1.000000e-01,1.450908e+05,375.0,4.234,63.0,1.629,90.0,2.327,78.0,2.017,49.0,1.27,5.62,0.399,0.019,0.270,2.402,0.55,2.74,160.5,2.04,11.93,217.15
3,34.0,3.0,30995.282610,39988.452940,179.0,1.0,1.33,82013.365563,2.000000e+00,5.397605e-79,15.0,15.0,30.0,30.0,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0,90.6,173.3,30.2,42.8,36.2,35.7,106.1,110.6,10.0,11.0,3.1,0.3,9.7,163.0,56.0,4.60,4.60,113.0,9989.2,4.07,44526.214135,5.5,54.4,7.5,35.6,2.1,0.5,3.0,0.4,2.0,1.000000e-01,5.397605e-79,5.06,15.4,43.5,86.1,35.3,30.4,12.8,250.0,7.4,5.397605e-79,8.259962e+04,142.0,1.603,109.0,2.819,111.0,2.870,112.0,2.896,46.0,1.19,1.05,1.810,0.087,0.190,1.690,0.34,1.70,183.3,2.33,10.94,199.13
4,51.0,4.0,41925.463225,51305.024430,174.0,2.0,5.00,57902.412643,1.000000e+00,5.397605e-79,15.0,15.0,30.0,5.0,99.0,69.0,110.0,68.0,123.0,67.0,78.0,82.0,79.0,76.7,177.3,24.4,41.0,38.2,29.5,92.1,99.4,10.0,10.0,3.8,0.4,10.5,217.0,76.0,3.02,3.02,48.0,4243.2,6.29,52478.876664,6.6,18.3,10.3,70.4,0.7,0.5,1.2,0.7,4.6,5.397605e-79,5.397605e-79,4.90,14.7,42.6,87.0,34.5,30.0,12.9,232.0,9.0,4.000000e-01,1.004203e+05,57.0,0.644,124.0,3.207,120.0,3.103,124.0,3.207,48.0,1.24,0.92,0.345,0.017,0.129,1.148,0.27,1.35,185.0,2.35,9.36,170.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2522,54.0,2.0,26903.973076,31381.453127,185.0,1.0,3.42,32484.337247,1.000

In [18]:
# Probar esto para ver que los min/max son correctos
min_values_num = [min_values_dict[col] for col in df_num.columns]
max_values_num = [max_values_dict[col] for col in df_num.columns]
min_values_num
max_values_num

[69.0,
 7.0,
 170968.343177,
 227108.296958,
 187.0,
 2.0,
 5.0,
 408505.813642,
 99.0,
 77.0,
 15.0,
 15.0,
 30.0,
 30.0,
 211.0,
 131.0,
 208.0,
 129.0,
 204.0,
 134.0,
 129.0,
 129.0,
 128.0,
 224.1,
 198.1,
 74.8,
 49.8,
 49.2,
 61.0,
 177.2,
 182.5,
 30.0,
 30.0,
 75.0,
 72.6,
 571.7,
 400.0,
 207.0,
 3325.46,
 3325.46,
 805.0,
 71162.0,
 5763.61,
 241728.8572411242,
 18.4,
 66.5,
 22.1,
 86.9,
 24.4,
 4.2,
 9.3,
 1.7,
 14.6,
 2.9,
 0.2,
 7.08,
 18.6,
 54.9,
 113.3,
 37.7,
 41.1,
 37.5,
 675.0,
 12.5,
 4.0,
 561922.4295631719,
 1745.0,
 19.701,
 314.0,
 8.12,
 313.0,
 8.094,
 314.0,
 8.12,
 131.0,
 3.39,
 106.78,
 48.07,
 2.322,
 7.684,
 68.365,
 29.54,
 147.4,
 752.4,
 9.56,
 42.95,
 781.78]

In [ ]:
#min_values_all = [min_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
#max_values_all = [max_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
#max_values_all

### Importante 

Aunque MissForest solo modifica las variables con valores faltantes, el imputador necesita todas las columnas para predecir, por eso hay que establecer correctamente los rangos min_value y max_value para todas las columnas, incluso las que no tienen valores faltantes, para evitar imputaciones fuera de rango cuando esas columnas sirven como predictores.

### Función cuando hacemos split en train/test

In [20]:
train_num, test_num = train_test_split(
    df_num,
    test_size=0.2,
    random_state=1234
)

In [28]:
# Llamar a la función para imputar
train_final, test_final = imputacion_pipeline(
    train_num=train_num,
    test_num=test_num,
    #train_cat=train_cat,
    #test_cat=test_cat,
    num_estimators=50, # Número de árboles en el modelo ExtraTrees
    m_depth=4, # Profundidad máxima de los árboles
    m_samples_leaf=3, # Mínimo de muestras en cada hoja
    max_iter=100,
    random_state=1234,
    guardar_imputador=True
)

[IterativeImputer] Completing matrix with shape (2021, 87)
[IterativeImputer] Ending imputation round 1/100, elapsed time 12.79
[IterativeImputer] Change: 644.1612974587999, scaled tolerance: 561.9224295631719 
[IterativeImputer] Ending imputation round 2/100, elapsed time 24.64
[IterativeImputer] Change: 532.0261890115019, scaled tolerance: 561.9224295631719 
[IterativeImputer] Early stopping criterion reached.
[IterativeImputer] Completing matrix with shape (2021, 87)
[IterativeImputer] Ending imputation round 1/2, elapsed time 0.24
[IterativeImputer] Ending imputation round 2/2, elapsed time 0.49
[IterativeImputer] Completing matrix with shape (506, 87)
[IterativeImputer] Ending imputation round 1/2, elapsed time 0.20
[IterativeImputer] Ending imputation round 2/2, elapsed time 0.41
[IterativeImputer] Completing matrix with shape (2021, 87)
[IterativeImputer] Ending imputation round 1/100, elapsed time 11.88
[IterativeImputer] Change: 0.0, scaled tolerance: 561.9224295631719 
[Itera

### Función sin split en train/test

In [32]:
data_final = imputacion_pipeline_all(
    data_num=df_num,
    #data_cat=df_cat_ord,   # aquí SOLO ordinales (codificadas como enteros)
    num_estimators=50,
    m_depth=4,
    m_samples_leaf=3,
    max_iter=100,
    random_state=1234,
    guardar_imputador=True)

[IterativeImputer] Completing matrix with shape (2527, 87)
[IterativeImputer] Ending imputation round 1/100, elapsed time 15.12
[IterativeImputer] Change: 1138.9047460000543, scaled tolerance: 561.9224295631719 
[IterativeImputer] Ending imputation round 2/100, elapsed time 29.19
[IterativeImputer] Change: 585.091303453786, scaled tolerance: 561.9224295631719 
[IterativeImputer] Ending imputation round 3/100, elapsed time 43.45
[IterativeImputer] Change: 584.2228169692257, scaled tolerance: 561.9224295631719 
[IterativeImputer] Ending imputation round 4/100, elapsed time 57.63
[IterativeImputer] Change: 393.78567096591445, scaled tolerance: 561.9224295631719 
[IterativeImputer] Early stopping criterion reached.
[IterativeImputer] Completing matrix with shape (2527, 87)
[IterativeImputer] Ending imputation round 1/4, elapsed time 0.25
[IterativeImputer] Ending imputation round 2/4, elapsed time 0.50
[IterativeImputer] Ending imputation round 3/4, elapsed time 0.75
[IterativeImputer] End

### Imputación de las variables categóricas nominales usando la moda

In [33]:
def impute_nominal_simple(X_nom):
    imp = SimpleImputer(strategy='most_frequent')
    X_nom_imp = pd.DataFrame(imp.fit_transform(X_nom), columns=X_nom.columns, index=X_nom.index)
    return X_nom_imp, imp

In [34]:
X_nom_imp, imp = impute_nominal_simple(df_cat_nom)

### Unión de todas las variables

In [35]:
df_imp = pd.concat([data_final, X_nom_imp], axis=1)

In [36]:
# Comprobar que no hay valores faltantes
df_imp.isna().sum().sum()

0

In [37]:
df_imp

,edad,tamano_hogar,peso_entrevista,peso_examen,estrato,psu,ratio_pobreza,peso_dieta,total_suplementos,total_antiacidos,romberg_1_seg,romberg_2_seg,romberg_3_seg,romberg_4_seg,sistolica_1,diastolica_1,sistolica_2,diastolica_2,sistolica_3,diastolica_3,pulso_1,pulso_2,pulso_3,peso_kg,talla_cm,imc,pierna_cm,brazo_cm,perim_brazo,cintura_cm,cadera_cm,med_validas,intentos,rigidez_kpa,iqr_rigidez,ratio_iqrm,cap_dbm,iqr_cap,albumina_orina,albmina_si,creatinina_orina,creatinina_si,ratio_albcre,peso_sangre,blancos,_linfocitos,_monocitos,_neutrofilos,_eosinofilos,_basofilos,linfocitos_abs,monocitos_abs,neutrofilos_abs,eosinofilos_abs,basofilos_abs,rojos,hemoglobina,hematocrito,vcm,chcm,hcm,rdw,plaquetas,vpm,nrbc,peso_ayuno,trigliceridos,trigli_si,ldl,ldl_si,ldl_martin,ldl_martin_si,ldl_nih,ldl_nih_si,hdl,hdl_si,proteina_c,plomo,plomo_si,cadmio,cadmio_si,mercurio,mercurio_si,selenio,selenio_si,manganeso,manganeso_si,genero,etnia_1,etnia_3,mes_examen,pais_nacimiento,uso_suplementos,uso_antiacidos,estado_equilibrio,brazo,manguito,estado_antropo,estado_higado,sonda,com_albumina,com_pcr,com_cadmio,com_mercurio,movilidad,neurologico,osteoporosis,equilibrio
0,43.0,4.0,50055.450807,54374.463898,173.0,2.0,5.00,61366.555827,5.397605e-79,5.397605e-79,15.000000,15.000000,30.000000,13.000000,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0,86.9,179.5,27.0,42.8,42.0,35.7,98.3,102.9,10.0,11.0,9.5,2.3,24.2,380.0,19.0,23.12,23.12,136.0,12022.4,17.00,56042.129410,4.7,40.8,6.8,46.1,5.2,1.2,1.9,0.3,2.2,2.000000e-01,1.000000e-01,5.19,15.7,45.3,87.4,34.6,30.2,12.9,259.0,7.8,3.000000e-01,1.200253e+05,153.0,1.727,188.0,4.862,190.0,4.913,191.0,4.939,45.0,1.16,1.78,2.810,0.136,0.117,1.041,1.01,5.04,189.2,2.40,10.94,199.13,1.0,5.0,6.0,2.0,2.0,0.0,0.0,1.0,b'R',4.0,1.0,1.0,b'M',0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,66.0,2.0,29087.450605,34084.721548,173.0,2.0,5.00,34638.056480,5.397605e-79,5.397605e-79,15.000000,15.000000,30.000000,25.000000,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0,101.8,174.2,33.5,38.5,38.7,33.7,114.7,112.4,12.0,17.0,3.9,0.8,20.5,345.0,29.0,4.25,4.25,64.0,5657.6,6.64,37435.705647,6.3,25.4,14.2,57.3,2.5,0.7,1.6,0.9,3.6,2.000000e-01,5.397605e-79,4.59,15.2,43.9,95.8,34.5,33.0,12.8,221.0,8.5,1.000000e-01,5.397605e-79,86.0,0.971,137.0,3.543,135.0,3.491,139.0,3.595,60.0,1.55,2.03,2.040,0.099,0.313,2.785,9.64,48.10,192.3,2.44,7.74,140.88,1.0,3.0,3.0,2.0,1.0,0.0,0.0,1.0,b'R',4.0,1.0,1.0,b'XL',0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
2,44.0,7.0,80062.674301,81196.277992,174.0,1.0,1.41,84728.261560,1.000000e+00,5.397605e-79,15.000000,15.000000,30.000000,2.000000,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0,69.4,152.9,29.7,38.5,35.5,36.3,93.5,98.0,10.0,10.0,7.6,1.4,18.4,255.0,90.0,12.43,12.43,157.0,13878.8,7.92,85328.844519,5.7,29.9,5.5,58.5,5.3,0.9,1.7,0.3,3.3,3.000000e-01,1.000000e-01,4.86,13.8,40.1,82.5,34.4,28.3,13.3,235.0,9.1,1.000000e-01,1.450908e+05,375.0,4.234,63.0,1.629,90.0,2.327,78.0,2.017,49.0,1.27,5.62,0.399,0.019,0.270,2.402,0.55,2.74,160.5,2.04,11.93,217.15,2.0,2.0,2.0,1.0,2.0,1.0,0.0,1.0,b'R',4.0,1.0,1.0,b'M',0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
3,34.0,3.0,30995.282610,39988.452940,179.0,1.0,1.33,82013.365563,2.000000e+00,5.397605e-79,15.000000,15.000000,30.000000,30.000000,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0,90.6,173.3,30.2,42.8,36.2,35.7,106.1,110.6,10.0,11.0,3.1,0.3,9.7,163.0,56.0,4.60,4.60,113.0,9989.2,4.07,44526.214135,5.5,54.4,7.5,35.6,2.1,0.5,3.0,0.4,2.0,1.000000e-01,5.397605e-79,5.06,15.4,43.5,86.1,35.3,30.4,12.8,250.0,7.4,5.397605e-79,8.259962e+04,142.0,1.603,109.0,2.819,111.0,2.870,112.0,2.896,46.0,1.19,1.05,1.810,0.087,0.190,1.690,0.34,1.70,183.3,2.33,10.94,199.13,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,b'R',4.0,1.0,1.0,b'M',0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,51.0,4.0,41925.463225,51305.024430,174.0,2.0,5.00,57902.412643,1.000000e+00,5.397605e-79,15.000000,15.000000,30.000000,5.000000,99.0,69.0,110.0,68.0,123.0,67.0,78.0,82.0,79.0,76.7,177.3,24.4,41.0,38.2,29.5,92.1,99.4,10.0,10.0,3.8,0.4,10.5,217.0,76.0,3.02,3.02,48.0,4243.2,6.29,52478.87

## Comparación de la distribución

En esta parte lo que comparamos es:

- **Distribución original**: los valores no nulos originales 

- **Distribución imputada completa**: todos los valores del train tras imputar, que incluyen imputados + originales.

Así podemos responder preguntas como: **¿La imputación ha distorsionado de forma visible la forma de la distribución?**

In [58]:
# Estadísticas antes de la imputación
df.describe()

,genero,edad,etnia_1,etnia_3,mes_examen,pais_nacimiento,tamano_hogar,peso_entrevista,peso_examen,estrato,psu,ratio_pobreza,peso_dieta,total_suplementos,total_antiacidos,uso_suplementos,uso_antiacidos,estado_equilibrio,romberg_1_seg,romberg_2_seg,romberg_3_seg,romberg_4_seg,manguito,sistolica_1,diastolica_1,sistolica_2,diastolica_2,sistolica_3,diastolica_3,pulso_1,pulso_2,pulso_3,estado_antropo,peso_kg,talla_cm,imc,pierna_cm,brazo_cm,perim_brazo,cintura_cm,cadera_cm,estado_higado,med_validas,intentos,rigidez_kpa,iqr_rigidez,ratio_iqrm,cap_dbm,iqr_cap,albumina_orina,albmina_si,com_albumina,creatinina_orina,creatinina_si,ratio_albcre,peso_sangre,blancos,_linfocitos,_monocitos,_neutrofilos,_eosinofilos,_basofilos,linfocitos_abs,monocitos_abs,neutrofilos_abs,eosinofilos_abs,basofilos_abs,rojos,hemoglobina,hematocrito,vcm,chcm,hcm,rdw,plaquetas,vpm,nrbc,peso_ayuno,trigliceridos,trigli_si,ldl,ldl_si,ldl_martin,ldl_martin_si,ldl_nih,ldl_nih_si,hdl,hdl_si,proteina_c,com_pcr,plomo,plomo_si,cadmio,cadmio_si,com_cadmio,mercurio,mercurio_si,com_mercurio,selenio,selenio_si,manganeso,manganeso_si,movilidad,neurologico,osteoporosis,equilibrio
count,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2.214000e+03,2.527000e+03,1.995000e+03,1.990000e+03,1992.000000,1989.000000,2527.000000,2121.000000,2.117000e+03,2091.000000,2.078000e+03,2492.000000,2469.000000,2469.000000,2467.000000,2467.000000,2460.000000,2460.000000,2469.000000,2467.000000,2460.000000,2527.000000,2516.000000,2521.000000,2515.000000,2412.000000,2490.000000,2490.000000,2451.000000,2434.000000,2442.00000,2.428000e+03,2.527000e+03,2419.000000,2.416000e+03,2.416000e+03,2418.000000,2.415000e+03,2503.000000,2503.000000,2.503000e+03,2503.000000,2503.000000,2503.000000,2527.000000,2517.000000,2513.000000,2513.000000,2513.000000,2.513000e+03,2513.000000,2513.000000,2513.000000,2513.000000,2.513000e+03,2.513000e+03,2517.000000,2517.000000,2517.000000,2517.000000,2517.000000,2517.000000,2517.000000,2517.000000,2517.000000,2.513000e+03,2.527000e+03,2393.000000,2393.000000,2361.000000,2361.000000,2361.000000,2361.000000,2389.000000,2389.000000,2411.000000,2411.000000,2455.000000,2.455000e+03,2524.000000,2524.000000,2524.000000,2524.000000,2.524000e+03,2524.000000,2524.000000,2.524000e+03,2524.000000,2524.000000,2524.000000,2524.000000,2160.000000,2334.000000,2134.000000,2426.000000
mean,1.562327,48.621686,3.077562,3.256826,1.535022,1.225564,2.699644,33826.990888,43509.468658,179.671547,1.491888,3.002191e+00,4.402875e+04,1.874185e+00,2.201005e-01,0.619478,0.174962,1.480412,14.886374,1.480255e+01,29.624103,2.192637e+01,3.693820,119.897934,75.907655,120.005270,75.292663,119.765447,75.002846,70.801134,71.448723,71.969512,1.091413,84.204956,167.674336,29.898608,38.673217,37.606707,33.926386,100.432925,108.149712,0.93407,1.041310e+01,1.314009e+01,6.100083,1.101242e+00,1.583212e+01,262.677006,3.713540e+01,31.394934,31.394934,7.271274e-02,140.938474,12458.961087,27.672833,45650.229176,6.633294,30.595145,7.964743,57.948945,2.791604e+00,0.824990,1.967728,0.514922,3.924751,1.823319e-01,4.934341e-02,4.714418,14.036313,41.590902,88.459436,33.721812,29.843862,13.795113,259.810091,8.156218,7.389574e-02,8.225140e+04,121.478061,1.371489,112.321050,2.904640,113.463787,2.934176,114.543742,2.962107,54.413936,1.407366,3.996118,4.887984e-03,0.932163,0.045028,0.419276,3.730300,9.112520e-03,1.254782,6.261977,1.810618e-01,181.756894,2.308245,9.606668,174.860618,0.162963,0.988003,0.998126,0.875515
std,0.496198,14.397566,1.023245,1.415098,0.498871,0.418036,1.469811,22867.286454,31066.251699,4.243825,0.500033,1.695412e+00,4.751253e+04,3.936776e+00,1.768616e+00,0.485637,0.380030,1.022467,1.121744,1.431397e+00,2.846348,1.125619e+01,0.608182,16.846411,11.031724,16.773199,11.017142,16.795128,10.934516,12.190535,12.186278,12.275022,0.408677,22.904717,9.919946,7.571434,3.627766,2.883551,5.283013,17.121122,14.413470,0.24821,1.700077e+00,5.99123

In [39]:
df_imp.describe()

,edad,tamano_hogar,peso_entrevista,peso_examen,estrato,psu,ratio_pobreza,peso_dieta,total_suplementos,total_antiacidos,romberg_1_seg,romberg_2_seg,romberg_3_seg,romberg_4_seg,sistolica_1,diastolica_1,sistolica_2,diastolica_2,sistolica_3,diastolica_3,pulso_1,pulso_2,pulso_3,peso_kg,talla_cm,imc,pierna_cm,brazo_cm,perim_brazo,cintura_cm,cadera_cm,med_validas,intentos,rigidez_kpa,iqr_rigidez,ratio_iqrm,cap_dbm,iqr_cap,albumina_orina,albmina_si,creatinina_orina,creatinina_si,ratio_albcre,peso_sangre,blancos,_linfocitos,_monocitos,_neutrofilos,_eosinofilos,_basofilos,linfocitos_abs,monocitos_abs,neutrofilos_abs,eosinofilos_abs,basofilos_abs,rojos,hemoglobina,hematocrito,vcm,chcm,hcm,rdw,plaquetas,vpm,nrbc,peso_ayuno,trigliceridos,trigli_si,ldl,ldl_si,ldl_martin,ldl_martin_si,ldl_nih,ldl_nih_si,hdl,hdl_si,proteina_c,plomo,plomo_si,cadmio,cadmio_si,mercurio,mercurio_si,selenio,selenio_si,manganeso,manganeso_si
count,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2.527000e+03,2.527000e+03,2.527000e+03,2.527000e+03,2527.000000,2.527000e+03,2527.000000,2.527000e+03,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2.527000e+03,2.527000e+03,2527.000000,2.527000e+03,2.527000e+03,2527.000000,2.527000e+03,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2.527000e+03,2527.000000,2527.000000,2527.000000,2527.000000,2.527000e+03,2.527000e+03,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2.527000e+03,2.527000e+03,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000,2527.000000
mean,48.621686,2.699644,33826.990888,43509.468658,179.671547,1.491888,3.001904e+00,4.402875e+04,2.800279e+00,2.496506e-01,14.879127,1.472657e+01,29.595703,2.177961e+01,119.826477,75.890397,119.940694,75.269970,119.700262,74.978343,70.778697,71.415791,71.919424,84.207868,167.675784,29.899752,38.672906,37.603897,33.946697,100.621931,108.357589,1.036876e+01,1.314009e+01,6.096164,1.091404e+00,1.576833e+01,263.205186,3.710106e+01,31.255926,31.256133,140.850025,12451.108888,27.558343,45650.229176,6.633035,30.592943,7.966300,57.949375,2.791899e+00,0.825051,1.967351,0.514903,3.923550,1.823249e-01,4.933102e-02,4.714237,14.036605,41.591236,88.462185,33.722142,29.844837,13.793911,259.801449,8.156234,7.386598e-02,8.225140e+04,120.487091,1.360301,112.261154,2.903068,113.397752,2.932477,114.513460,2.961322,54.454654,1.408533,4.015766,0.932077,0.045023,0.419191,3.729539,1.254520,6.260709,181.758698,2.308267,9.606054,174.849441
std,14.397566,1.469811,22867.286454,31066.251699,4.243825,0.500033,1.592705e+00,4.751253e+04,4.017242e+00,1.586934e+00,1.029247,1.326577e+00,2.594013,1.033080e+01,16.658429,10.904923,16.585739,10.891516,16.609484,10.811127,12.050703,12.043535,12.138254,22.872133,9.908893,7.558152,3.579572,2.868297,5.305477,17.302777,14.674705,1.680955e+00,5.991232e+00,5.421742,2.654440e+00,1.820718e+01,62.821574,2.053036e+01,140.164510,140.164535,91.745138,8110.275751,170.416481,33364.659419,2.019403,8.320452,2.072415,9.093208,2.039483e+00,0.340008,0.648188,0.171615,1.601558,1.516548e-01,5.120057e-02,0.472799,1.456057,3.989827,5.675859,0.859350,2.290557,1.382989,67.090506,0.906093,1.081194e-01,6.960445e+04,90.420621,1.020843,35.423450,0.916080,34.818121,0.900411,35.695405,0.923083,14.245749,0.368457,7.202873,1.348623,0.065137,0.550367,4.896624,2.075988,10.358864,28.329013,0.359814,3.527548,64.208508
min,20.000000,1.000000,4584.463196,5424.587296,173.000000,1.000000,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,1.000000,5.397605e-79,1.000000,5.397605e-79,75.000000,44.000000,65.000000,39

### A partir de aquí ya vendría:

- Reagrupación de categorías poco frecuentes **NO**
- One-Hot para categóricas nominales
- Transformación de variables **AUN NO**
- Creación de variables derivadas **AUN NO**
- Normalización/estandarización de variables numéricas

In [86]:
df_model = df_imp.copy()

#### Reagrupación

In [87]:
umbral = 0.01

for col in cat_cols_nom:
    
    if col in df_model.columns:
        
        freq = df_model[col].value_counts(normalize=True)
        
        categorias_poco_frecuentes = freq[freq < umbral].index
        
        df_model[col] = df_model[col].replace(
            categorias_poco_frecuentes,
            -1
        )

C:\Users\Usuario\AppData\Local\Temp\ipykernel_23180\1667455573.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model[col] = df_model[col].replace(
C:\Users\Usuario\AppData\Local\Temp\ipykernel_23180\1667455573.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model[col] = df_model[col].replace(
C:\Users\Usuario\AppData\Local\Temp\ipykernel_23180\1667455573.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.

In [88]:
for col in cat_cols_nom:
    
    if col in df_model.columns:
        
        print(f"\n--- {col} ---")
        print(df_model[col].value_counts())


--- genero ---
genero
2.0    1421
1.0    1106
Name: count, dtype: int64

--- etnia_1 ---
etnia_1
3.0    1421
4.0     304
5.0     303
2.0     284
1.0     215
Name: count, dtype: int64

--- etnia_3 ---
etnia_3
3.0    1421
4.0     304
2.0     284
1.0     215
6.0     153
7.0     150
Name: count, dtype: int64

--- mes_examen ---
mes_examen
2.0    1352
1.0    1175
Name: count, dtype: int64

--- pais_nacimiento ---
pais_nacimiento
1.0    1957
2.0     570
Name: count, dtype: int64

--- uso_suplementos ---
uso_suplementos
1.0    1769
0.0     758
Name: count, dtype: int64

--- uso_antiacidos ---
uso_antiacidos
0.0    2179
1.0     348
Name: count, dtype: int64

--- estado_equilibrio ---
estado_equilibrio
1.0    2018
4.0     300
3.0     105
2.0     104
Name: count, dtype: int64

--- brazo ---
brazo
b'R'    2491
b''       29
-1         7
Name: count, dtype: int64

--- manguito ---
manguito
 4.0    1384
 3.0     947
 5.0     192
-1.0       4
Name: count, dtype: int64

--- estado_antropo ---
estado_

#### One-hot encoding

In [89]:
df_model["brazo"] = df_model["brazo"].astype(str)
df_model["sonda"] = df_model["sonda"].astype(str)

In [90]:
df_model = pd.get_dummies(
    df_model,
    columns=cat_cols_nom,
    drop_first=True,
    dtype=int
)

In [91]:
print("Shape original:", df_imp.shape)
print("Shape tras One-Hot:", df_model.shape)

df_model.head()

Shape original: (2527, 108)
Shape tras One-Hot: (2527, 123)


,edad,tamano_hogar,peso_entrevista,peso_examen,estrato,psu,ratio_pobreza,peso_dieta,total_suplementos,total_antiacidos,romberg_1_seg,romberg_2_seg,romberg_3_seg,romberg_4_seg,sistolica_1,diastolica_1,sistolica_2,diastolica_2,sistolica_3,diastolica_3,pulso_1,pulso_2,pulso_3,peso_kg,talla_cm,imc,pierna_cm,brazo_cm,perim_brazo,cintura_cm,cadera_cm,med_validas,intentos,rigidez_kpa,iqr_rigidez,ratio_iqrm,cap_dbm,iqr_cap,albumina_orina,albmina_si,creatinina_orina,creatinina_si,ratio_albcre,peso_sangre,blancos,_linfocitos,_monocitos,_neutrofilos,_eosinofilos,_basofilos,linfocitos_abs,monocitos_abs,neutrofilos_abs,eosinofilos_abs,basofilos_abs,rojos,hemoglobina,hematocrito,vcm,chcm,hcm,rdw,plaquetas,vpm,nrbc,peso_ayuno,trigliceridos,trigli_si,ldl,ldl_si,ldl_martin,ldl_martin_si,ldl_nih,ldl_nih_si,hdl,hdl_si,proteina_c,plomo,plomo_si,cadmio,cadmio_si,mercurio,mercurio_si,selenio,selenio_si,manganeso,manganeso_si,genero_2.0,etnia_1_2.0,etnia_1_3.0,etnia_1_4.0,etnia_1_5.0,etnia_3_2.0,etnia_3_3.0,etnia_3_4.0,etnia_3_6.0,etnia_3_7.0,mes_examen_2.0,pais_nacimiento_2.0,uso_suplementos_1.0,uso_antiacidos_1.0,estado_equilibrio_2.0,estado_equilibrio_3.0,estado_equilibrio_4.0,brazo_b'',brazo_b'R',manguito_3.0,manguito_4.0,manguito_5.0,estado_antropo_1.0,estado_antropo_2.0,estado_antropo_3.0,estado_higado_1.0,sonda_b'M',sonda_b'XL',com_albumina_1.0,com_pcr_5.397605346934028e-79,com_cadmio_5.397605346934028e-79,com_mercurio_1.0,movilidad_1.0,neurologico_1.0,osteoporosis_1.0,equilibrio_1.0
0,43.0,4.0,50055.450807,54374.463898,173.0,2.0,5.00,61366.555827,5.397605e-79,5.397605e-79,15.0,15.0,30.0,13.0,135.0,98.0,131.0,96.0,132.0,94.0,82.0,79.0,82.0,86.9,179.5,27.0,42.8,42.0,35.7,98.3,102.9,10.0,11.0,9.5,2.3,24.2,380.0,19.0,23.12,23.12,136.0,12022.4,17.00,56042.129410,4.7,40.8,6.8,46.1,5.2,1.2,1.9,0.3,2.2,2.000000e-01,1.000000e-01,5.19,15.7,45.3,87.4,34.6,30.2,12.9,259.0,7.8,3.000000e-01,1.200253e+05,153.0,1.727,188.0,4.862,190.0,4.913,191.0,4.939,45.0,1.16,1.78,2.810,0.136,0.117,1.041,1.01,5.04,189.2,2.40,10.94,199.13,0,0,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,1,0,1,0,1,0,0,1,1,0,0,1,1,0,0,1,1,1
1,66.0,2.0,29087.450605,34084.721548,173.0,2.0,5.00,34638.056480,5.397605e-79,5.397605e-79,15.0,15.0,30.0,25.0,121.0,84.0,117.0,76.0,113.0,76.0,72.0,71.0,73.0,101.8,174.2,33.5,38.5,38.7,33.7,114.7,112.4,12.0,17.0,3.9,0.8,20.5,345.0,29.0,4.25,4.25,64.0,5657.6,6.64,37435.705647,6.3,25.4,14.2,57.3,2.5,0.7,1.6,0.9,3.6,2.000000e-01,5.397605e-79,4.59,15.2,43.9,95.8,34.5,33.0,12.8,221.0,8.5,1.000000e-01,5.397605e-79,86.0,0.971,137.0,3.543,135.0,3.491,139.0,3.595,60.0,1.55,2.03,2.040,0.099,0.313,2.785,9.64,48.10,192.3,2.44,7.74,140.88,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,1,0,1,0,1,1,0,0,1,1,1
2,44.0,7.0,80062.674301,81196.277992,174.0,1.0,1.41,84728.261560,1.000000e+00,5.397605e-79,15.0,15.0,30.0,2.0,111.0,79.0,112.0,80.0,104.0,76.0,84.0,83.0,77.0,69.4,152.9,29.7,38.5,35.5,36.3,93.5,98.0,10.0,10.0,7.6,1.4,18.4,255.0,90.0,12.43,12.43,157.0,13878.8,7.92,85328.844519,5.7,29.9,5.5,58.5,5.3,0.9,1.7,0.3,3.3,3.000000e-01,1.000000e-01,4.86,13.8,40.1,82.5,34.4,28.3,13.3,235.0,9.1,1.000000e-01,1.450908e+05,375.0,4.234,63.0,1.629,90.0,2.327,78.0,2.017,49.0,1.27,5.62,0.399,0.019,0.270,2.402,0.55,2.74,160.5,2.04,11.93,217.15,1,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,1,0,1,0,1,0,0,1,1,0,0,1,1,0,1,1,1,1
3,34.0,3.0,30995.282610,39988.452940,179.0,1.0,1.33,82013.365563,2.000000e+00,5.397605e-79,15.0,15.0,30.0,30.0,110.0,72.0,120.0,74.0,115.0,75.0,59.0,64.0,64.0,90.6,173.3,30.2,42.8,36.2,35.7,106.1,110.6,10.0,11.0,3.1,0.3,9.7,163.0,56.0,4.60,4.60,113.0,9989.2,4.07,44526.214135,5.5,54.4,7.5,35.6,2.1,0.5,3.0,0.4,2.0,1.000000e-01,5.397605e-79,5.06,15.4,43.5,86.1,35.3,30.4,12.8,250.0,7.4,5.397605e-79,8.259962e+04,142.0,1.603,109.0,2.819,111.0,2.870,112.0,2.896,46.0,1.19,1.05,1.810,0.087,0.190,1.690,0.34,1.70,183.3,2.33,10.94,199.13,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,1,0,0,1,1,0,0,1,1,0,0,1,1,1
4,51.0,4.0,41925.463225,51305.024430,174.0,2.0,5.00,57902.412643,1.000000e+00,5.397605e-

#### Transformación variables

**Revisar si esta transformación es realmente útil o mejor ignorar**

In [85]:
for col in num_cols:
    
    if col in df_model.columns:
        
        if (df_model[col] >= 0).all():
            
            df_model[col + "_log"] = np.log1p(df_model[col])

C:\Users\Usuario\AppData\Local\Temp\ipykernel_23180\1975950717.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_model[col + "_log"] = np.log1p(df_model[col])
C:\Users\Usuario\AppData\Local\Temp\ipykernel_23180\1975950717.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_model[col + "_log"] = np.log1p(df_model[col])
C:\Users\Usuario\AppData\Local\Temp\ipykernel_23180\1975950717.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perf

In [69]:
print("Variables transformadas:\n")

for col in df_num:
    
    if col + "_log" in df_model.columns:
        
        print(f"\n--- {col} ---")
        
        print(
            df_model[
                [col, col + "_log"]
            ].head()
        )

Variables transformadas:


--- edad ---
   edad  edad_log
0  43.0  3.784190
1  66.0  4.204693
2  44.0  3.806662
3  34.0  3.555348
4  51.0  3.951244

--- tamano_hogar ---
   tamano_hogar  tamano_hogar_log
0           4.0          1.609438
1           2.0          1.098612
2           7.0          2.079442
3           3.0          1.386294
4           4.0          1.609438

--- peso_entrevista ---
   peso_entrevista  peso_entrevista_log
0     50055.450807            10.820907
1     29087.450605            10.278096
2     80062.674301            11.290578
3     30995.282610            10.341623
4     41925.463225            10.643672

--- peso_examen ---
    peso_examen  peso_examen_log
0  54374.463898        10.903668
1  34084.721548        10.436634
2  81196.277992        11.304637
3  39988.452940        10.596371
4  51305.024430        10.845563

--- estrato ---
   estrato  estrato_log
0    173.0     5.159055
1    173.0     5.159055
2    174.0     5.164786
3    179.0     5.192957
4    

#### Variables Derivadas

In [92]:
# Media presión sistólica
df_model["media_sistolica"] = df_model[
    ["sistolica_1", "sistolica_2", "sistolica_3"]
].mean(axis=1)

# Media presión diastólica
df_model["media_diastolica"] = df_model[
    ["diastolica_1", "diastolica_2", "diastolica_3"]
].mean(axis=1)

# Presión de pulso
df_model["presion_pulso"] = (
    df_model["media_sistolica"] -
    df_model["media_diastolica"]
)

In [93]:
df_model[
    [
        "media_sistolica",
        "media_diastolica",
        "presion_pulso"
    ]
].describe()

,media_sistolica,media_diastolica,presion_pulso
count,2527.000000,2527.000000,2527.000000
mean,119.822477,75.379570,44.442908
std,16.215929,10.589848,11.553865
min,77.668796,41.000000,16.666667
25%,109.000000,68.333333,36.666667
50%,117.333333,74.587332,42.666667
75%,128.166667,81.333333,50.000000
max,202.333333,131.000000,118.666667


#### Normalización y estandarización

- Hacer transform al test y fit al train!!

In [ ]:
from sklearn.preprocessing import StandardScaler

cols_numericas = df_model.select_dtypes(
    include=["int64", "float64"]
).columns

scaler = StandardScaler()

# Fit en train, transform en test
df_model[cols_numericas] = scaler.fit_transform(
    df_model[cols_numericas]
)

In [ ]:
df_model[num_cols].describe().T[
    ["mean", "std", "min", "max"]
].head(10)

,mean,std,min,max
edad,-1.799554e-16,1.000198,-1.988346,1.415680
tamano_hogar,-1.237194e-16,1.000198,-1.156598,2.926368
peso_entrevista,6.888918e-17,1.000198,-1.279046,5.998460
peso_examen,1.019279e-16,1.000198,-1.226167,5.911082
estrato,-6.748328e-17,1.000198,-1.572371,1.727193
psu,1.209075e-16,1.000198,-0.983905,1.016359
ratio_pobreza,4.498886e-17,1.000198,-1.885156,1.254778
peso_dieta,2.811803e-18,1.000198,-0.926860,7.672696
total_suplementos,7.170099e-17,1.000198,-0.697203,23.951450
total_antiacidos,1.968262e-17,1.000198,-0.157347,48.373493


: 